# ACTIVE-TB-S: Full Disagreement-Directed Third-Family Repair

This phase invokes CodeGemma only for tasks whose Qwen/Llama executable oracles have inadequate trace coverage or unresolved family disagreement. It receives sanitized specifications and generated evidence only. Golden RTL, official tests, and benchmark mutants remain absent.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HOME']='/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/kaggle/working/hf_cache'
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
markers=list(INPUT.rglob('active_tb_s_disagreement_repair_full_v1/REPAIR_BOUNDARY.json'))
if len(markers)==1: SOURCE=markers[0].parent
else:
 archives=list(INPUT.rglob('active_tb_s_disagreement_repair_full_v1.zip')); assert len(archives)==1,archives
 attached=WORK/'attached_full_repair_source'
 with zipfile.ZipFile(archives[0]) as zf: zf.extractall(attached)
 SOURCE=attached/'active_tb_s_disagreement_repair_full_v1'
PROJECT=WORK/'active_tb_s_repair_full_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project',PROJECT)
OUT=WORK/'active_tb_s_repaired_oracles_full_v1'; OUT.mkdir(exist_ok=True)
SNAPSHOT=WORK/'active_tb_s_repair_full_checkpoint.zip'
resume_archives=list(INPUT.rglob('active_tb_s_repair_full_checkpoint.zip'))
if len(resume_archives)>1: raise RuntimeError(resume_archives)
if resume_archives:
 resume_dir=WORK/'resume_full_repair'
 with zipfile.ZipFile(resume_archives[0]) as zf: zf.extractall(resume_dir)
 for name in ['codegemma_candidates.jsonl','repair_selection.json']:
  matches=list(resume_dir.rglob(name))
  if len(matches)==1: shutil.copy2(matches[0],OUT/name)
print('Isolated repair source:',SOURCE,'resume:',resume_archives)


In [ ]:
boundary=json.loads((SOURCE/'REPAIR_BOUNDARY.json').read_text())
assert boundary['golden_artifacts_in_dataset'] is False
assert not list(SOURCE.rglob('*mutant*'))
spec_rows=[json.loads(line) for line in (SOURCE/'spec_tasks_full.jsonl').read_text().splitlines() if line.strip()]
assert len(spec_rows)==156 and all(set(row)=={'task_id','description','header'} for row in spec_rows)
for name,expected in [('qwen_candidates.jsonl',312),('llama_candidates.jsonl',468)]:
 rows=[json.loads(line) for line in (SOURCE/name).read_text().splitlines() if line.strip()]
 assert len(rows)==expected
print('FULL_REPAIR_LEAKAGE_AUDIT_PASS:',len(spec_rows),'tasks')


In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','--force-reinstall',
 'torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','torchvision','torchaudio'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir',
 'transformers==4.57.6','accelerate>=1.0,<2','bitsandbytes==0.48.2','sentencepiece','tqdm>=4.66'],check=True)
probe=subprocess.run([sys.executable,'-c',
 'import torch; print(torch.__version__,torch.cuda.get_device_name(0)); x=torch.ones(1,device="cuda"); print("CUDA_PROBE_PASS",x.item())'],
 text=True,capture_output=True)
print(probe.stdout)
if probe.returncode: raise RuntimeError(probe.stderr)
subprocess.run([sys.executable,str(PROJECT/'spec_only_smoke_test.py')],check=True,cwd=PROJECT)


In [ ]:
MODEL='unsloth/codegemma-7b-it-bnb-4bit'
CODEGEMMA_OUT=OUT/'codegemma_candidates.jsonl'; ANALYSIS_OUT=OUT/'repair_selection.json'
cmd=[sys.executable,str(PROJECT/'repair_spec_oracles.py'),
 '--tasks',str(SOURCE/'spec_tasks_full.jsonl'),
 '--candidates',str(SOURCE/'qwen_candidates.jsonl'),str(SOURCE/'llama_candidates.jsonl'),
 '--output',str(CODEGEMMA_OUT),'--analysis-output',str(ANALYSIS_OUT),
 '--model',MODEL,'--family','codegemma','--candidates-per-task','2',
 '--resolution-threshold','0.85','--scenario-pool-size','96',
 '--max-new-tokens','900','--max-input-tokens','7168','--temperature','0.20',
 '--repair-attempts','1','--resume','--snapshot-path',str(SNAPSHOT),
 '--snapshot-root',str(OUT),'--snapshot-every','8']
subprocess.run(cmd,check=True,cwd=PROJECT)


In [ ]:
for name in ['qwen_candidates.jsonl','llama_candidates.jsonl']:
 shutil.copy2(SOURCE/name,OUT/name)
selection=json.loads((OUT/'repair_selection.json').read_text())
selected={row['task_id'] for row in selection if row['selected']}
rows=[json.loads(line) for line in CODEGEMMA_OUT.read_text().splitlines() if line.strip()]
assert len(rows)==2*len(selected),(len(rows),len(selected))
assert {(row['task_id'],row['candidate_index']) for row in rows}=={(task_id,index) for task_id in selected for index in range(2)}
print('Selected tasks:',len(selected),'/ 156')
print('CodeGemma compiled:',sum(bool(row['compiled']) for row in rows),'/',len(rows))
manifest={'protocol':'full disagreement-directed third-family repair','task_count':156,
 'selected_task_count':len(selected),'model':MODEL,'golden_artifacts_seen':False,
 'candidate_files':['qwen_candidates.jsonl','llama_candidates.jsonl','codegemma_candidates.jsonl']}
(OUT/'repair_manifest.json').write_text(json.dumps(manifest,indent=2))
cache=Path(os.environ['HF_HOME'])
if cache.exists(): shutil.rmtree(cache)
archive=WORK/'active_tb_s_repaired_oracles_full_v1.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',OUT)
print('Augmented full oracle checkpoint:',archive,archive.stat().st_size,'bytes')
